- load config
- load fact table
- normalize dataframe
- validate the schema
- save to sqlite
- run check queries

`loader`, `schema`, `normalizer`, `db`

In [41]:
import os
import re
import sqlite3
from pathlib import Path
import pandas as pd
import yaml
from datetime import datetime
from typing import Dict, Any



### load config

In [42]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "02_configs").exists() and (candidate / "00_data").exists():
            return candidate
    raise FileNotFoundError("could not find project root containing 02_configs and 00_data")


def load_yaml(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    return data or {}





def resolve_existing_path(base_root: Path, config_dir: Path, raw_path: str) -> Path:
    p = Path(str(raw_path))
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.append((base_root / p).resolve())
        candidates.append((config_dir / p).resolve())

        p_str = str(p).replace("\\", "/")
        while p_str.startswith("../"):
            p_str = p_str[3:]
        if p_str:
            candidates.append((base_root / p_str).resolve())

    for c in candidates:
        if c.exists():
            return c

    raise FileNotFoundError(f"path not found for '{raw_path}'. tried: {[str(c) for c in candidates]}")




In [43]:

ROOT = find_project_root(Path.cwd().resolve())
CONFIG_DIR = ROOT / "02_configs"

# load yaml files

app_config = load_yaml(CONFIG_DIR / "app_config.yaml")
table_registry = load_yaml(CONFIG_DIR / "table_regis.yaml")
vocabulary = load_yaml(CONFIG_DIR / "vocabulary.yaml")
metric_catalog = load_yaml(CONFIG_DIR / "metric_catalog.yaml")
data_rules = load_yaml(CONFIG_DIR / "data_rules.yaml")
unsupported_topic = load_yaml(CONFIG_DIR / "unsupported_topic.yaml")

active_table = app_config.get("tables", {}).get("active_table", "historyCalls")
history_calls_cfg = table_registry.get("tables", {}).get(active_table, {})

csv_path_raw = history_calls_cfg.get("data_path", "00_data/historyCalls/03_processed/fact_history_calls_v3.csv")
csv_path = resolve_existing_path(ROOT, CONFIG_DIR, csv_path_raw)


# sqlite
sqlite_path_raw = app_config.get("database", {}).get("sqlite_path", "data/historyCalls/03_processed/historyassistant.db")
sqlite_db_path = (ROOT / sqlite_path_raw).resolve()

defaults = app_config.get("defaults", {})


print(f"ROOT: {ROOT}")
print(f"active table: {active_table}")
print(f"CSV path: {csv_path}")
print(f"sqlite DB path: {sqlite_db_path}")
print(f"default policies: {defaults}")

ROOT: C:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant
active table: historyCalls
CSV path: C:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant\00_data\historyCalls\03_processed\fact_history_calls_v3.csv
sqlite DB path: C:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant\data\historyCalls\03_processed\historyassistant.db
default policies: {'scope_policy': 'qa_ready', 'outlier_policy': 'exclude_negative'}


### load fact table historyCalls

In [44]:
fact_df = pd.read_csv(csv_path)
print("load shape:", fact_df.shape)

display(fact_df.head(3))

load shape: (36495, 39)


,call_id,timestamp_utc,cancel_timestamp_utc,call_date,call_hour,call_dayofweek,shift,call_type,priority,ward_id,...,dq_extreme_response_time_flag,dq_invalid_priority_flag,dq_invalid_hour_flag,dq_invalid_dayofweek_flag,dq_cancelled_missing_person_flag,dq_unknown_person_type_flag,dq_negative_response_time_flag,dq_extreme_staying_time_flag,dq_parse_error_flag,qa_ready_flag
0,72733,2024-01-31 03:49:14+00:00,2024-01-31 04:08:10+00:00,31/01/2024,3,2,night,Normal,3,6,...,False,False,False,False,False,True,False,False,False,True
1,72734,2024-01-31 03:51:47+00:00,2024-01-31 04:06:08+00:00,31/01/2024,3,2,night,Normal,3,6,...,False,False,False,False,False,True,False,False,False,True
2,72735,2024-01-31 04:21:31.109643100+00:00,2024-01-31 04:21:55.072143600+00:00,31/01/2024,4,2,night,Care,3,-1,...,False,False,False,False,False,False,False,False,False,True


In [45]:
print("columns:")
for col in fact_df.columns:
    print("-", col)

print("\ndtypes:")
print(fact_df.dtypes)

columns:
- call_id
- timestamp_utc
- cancel_timestamp_utc
- call_date
- call_hour
- call_dayofweek
- shift
- call_type
- priority
- ward_id
- ward_name
- room_name
- zone_name
- area_name
- person_location
- person_type
- person_name
- call_name
- cancel_person
- cancellation_trigger
- triggered_by
- cancel_location
- is_cancelled
- response_time_sec
- staying_time_sec
- staying_time_min
- dq_unknown_location_flag
- dq_missing_staying_time_when_cancelled
- dq_missing_response_time
- dq_extreme_response_time_flag
- dq_invalid_priority_flag
- dq_invalid_hour_flag
- dq_invalid_dayofweek_flag
- dq_cancelled_missing_person_flag
- dq_unknown_person_type_flag
- dq_negative_response_time_flag
- dq_extreme_staying_time_flag
- dq_parse_error_flag
- qa_ready_flag

dtypes:
call_id                                     int64
timestamp_utc                                 str
cancel_timestamp_utc                          str
call_date                                     str
call_hour                   

### normlization

In [46]:
def normalize_fact_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    
    # standardize column names
    out.columns = [c.strip().lower() for c in out.columns]

    for col in ["timestamp_utc", "cancel_timestamp_utc"]:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce", utc=True)

    if "call_date" in out.columns:
        out["call_date"] = pd.to_datetime(out["call_date"], format="%d/%m/%Y", errors="coerce")

    # convert numeric columns
    numeric_cols = ["call_id", "call_hour", "call_dayofweek", "priority", "ward_id","response_time_sec", "staying_time_sec", "staying_time_min",]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")


    # normalize boolean values
    bool_like_cols = [c for c in out.columns if c.startswith("dq_")] + ["is_cancelled", "qa_ready_flag"]
    for col in bool_like_cols:
        if col in out.columns:
            out[col] = (
                out[col]
                .astype(str)
                .str.strip()
                .str.upper()
                .map({"TRUE": True, "FALSE": False})
            )

    # time date helper fields
    if "response_time_sec" in out.columns:
        out["is_response_time_valid"] = out["response_time_sec"].notna() & (out["response_time_sec"] >= 0)
    if "staying_time_sec" in out.columns:
        out["is_staying_time_valid"] = out["staying_time_sec"].notna() & (out["staying_time_sec"] >= 0)
    if "call_date" in out.columns:
        out["call_date_str"] = out["call_date"].dt.strftime("%Y-%m-%d")
    return out

fact_norm = normalize_fact_dataframe(fact_df)
print("normalized shape:", fact_norm.shape)

display(fact_norm.head(3))

normalized shape: (36495, 42)


,call_id,timestamp_utc,cancel_timestamp_utc,call_date,call_hour,call_dayofweek,shift,call_type,priority,ward_id,...,dq_invalid_dayofweek_flag,dq_cancelled_missing_person_flag,dq_unknown_person_type_flag,dq_negative_response_time_flag,dq_extreme_staying_time_flag,dq_parse_error_flag,qa_ready_flag,is_response_time_valid,is_staying_time_valid,call_date_str
0,72733,2024-01-31 03:49:14+00:00,2024-01-31 04:08:10+00:00,2024-01-31,3,2,night,Normal,3,6,...,False,False,True,False,False,False,True,True,True,2024-01-31
1,72734,2024-01-31 03:51:47+00:00,2024-01-31 04:06:08+00:00,2024-01-31,3,2,night,Normal,3,6,...,False,False,True,False,False,False,True,True,True,2024-01-31
2,72735,NaT,NaT,2024-01-31,4,2,night,Care,3,-1,...,False,False,False,False,False,False,True,True,True,2024-01-31


### validate

In [47]:
expected_required_cols = ["call_id", "timestamp_utc", "call_date", "call_type", "priority", "ward_name", "response_time_sec", "staying_time_sec", "is_cancelled",]

missing_cols = [c for c in expected_required_cols if c not in fact_norm.columns]
dtype_snapshot = fact_norm.dtypes.astype(str).to_dict()
bad_timestamp_count = 0

if "timestamp_utc" in fact_norm.columns:
    bad_timestamp_count = int(fact_norm["timestamp_utc"].isna().sum())

duplicate_call_id_count = 0

if "call_id" in fact_norm.columns:
    duplicate_call_id_count = int(fact_norm["call_id"].duplicated().sum())

null_rate = fact_norm.isna().mean().sort_values(ascending=False).round(4)



print("missing required columns:", missing_cols)
print("bad timestamp rows:", bad_timestamp_count)
print("duplicate call_id rows:", duplicate_call_id_count)
print("\ntop 15 null-rate columns:")

display(null_rate.head(15))

print("\ndatatype snapshot:")

for i, (k, v) in enumerate(dtype_snapshot.items()):
    if i >= 20:
        break
    print(f"- {k}: {v}")

missing required columns: []
bad timestamp rows: 20459
duplicate call_id rows: 0

top 15 null-rate columns:


cancel_timestamp_utc              0.5761
timestamp_utc                     0.5606
triggered_by                      0.0293
cancellation_trigger              0.0229
staying_time_min                  0.0229
staying_time_sec                  0.0229
cancel_location                   0.0229
cancel_person                     0.0229
call_name                         0.0140
person_type                       0.0121
dq_unknown_person_type_flag       0.0121
dq_negative_response_time_flag    0.0121
dq_extreme_staying_time_flag      0.0121
dq_parse_error_flag               0.0121
qa_ready_flag                     0.0121
dtype: float64


datatype snapshot:
- call_id: int64
- timestamp_utc: datetime64[us, UTC]
- cancel_timestamp_utc: datetime64[us, UTC]
- call_date: datetime64[us]
- call_hour: int64
- call_dayofweek: int64
- shift: str
- call_type: str
- priority: int64
- ward_id: int64
- ward_name: str
- room_name: str
- zone_name: str
- area_name: str
- person_location: str
- person_type: str
- person_name: str
- call_name: str
- cancel_person: str
- cancellation_trigger: str


### save to sqlite

In [48]:
sqlite_db_path.parent.mkdir(parents=True, exist_ok=True)
table_name = "fact_calls"

with sqlite3.connect(sqlite_db_path) as conn:
    fact_norm.to_sql(table_name, conn, if_exists="replace", index=False)

print(f"saved sqlite DB: {sqlite_db_path}")
print(f"saved table: {table_name}")

saved sqlite DB: C:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant\data\historyCalls\03_processed\historyassistant.db
saved table: fact_calls


### check query

In [49]:
queries = {
    "total_row_count": "SELECT COUNT(*) AS total_rows FROM fact_calls;",
    "min_max_date": "SELECT MIN(call_date) AS min_call_date, MAX(call_date) AS max_call_date FROM fact_calls;",
    "total_calls_per_day": "SELECT call_date, COUNT(*) AS call_count FROM fact_calls GROUP BY call_date ORDER BY call_date LIMIT 20;",
    "avg_response_time": "SELECT AVG(response_time_sec) AS avg_response_time_sec FROM fact_calls WHERE response_time_sec >= 0;",

}

with sqlite3.connect(sqlite_db_path) as conn:
    for name, sql in queries.items():
        print(f"\n=== {name} ===")
        result = pd.read_sql_query(sql, conn)

        display(result)


=== total_row_count ===


,total_rows
0,36495



=== min_max_date ===


,min_call_date,max_call_date
0,2024-01-31 00:00:00,2024-08-26 00:00:00



=== total_calls_per_day ===


,call_date,call_count
0,2024-01-31 00:00:00,170
1,2024-02-01 00:00:00,146
2,2024-02-02 00:00:00,182
3,2024-02-03 00:00:00,201
4,2024-02-04 00:00:00,224
5,2024-02-05 00:00:00,199
6,2024-02-06 00:00:00,223
7,2024-02-07 00:00:00,205
8,2024-02-08 00:00:00,223
9,2024-02-09 00:00:00,312



=== avg_response_time ===


,avg_response_time_sec
0,512.269589


In [50]:
query = """
SELECT
    call_date,
    COUNT(*) AS total_calls
FROM fact_calls
GROUP BY call_date
ORDER BY call_date
LIMIT 20;
"""
result = pd.read_sql_query(query, conn)
display(result)

,call_date,total_calls
0,2024-01-31 00:00:00,170
1,2024-02-01 00:00:00,146
2,2024-02-02 00:00:00,182
3,2024-02-03 00:00:00,201
4,2024-02-04 00:00:00,224
5,2024-02-05 00:00:00,199
6,2024-02-06 00:00:00,223
7,2024-02-07 00:00:00,205
8,2024-02-08 00:00:00,223
9,2024-02-09 00:00:00,312


In [51]:
query = """
SELECT
    MIN(call_date) AS min_call_date,
    MAX(call_date) AS max_call_date
FROM fact_calls;
"""
result = pd.read_sql_query(query, conn)
display(result)

,min_call_date,max_call_date
0,2024-01-31 00:00:00,2024-08-26 00:00:00


In [52]:
query = """
SELECT
    ward_name,
    COUNT(*) AS total_calls
FROM fact_calls
WHERE ward_name IS NOT NULL
GROUP BY ward_name
ORDER BY total_calls DESC
LIMIT 10;
"""
result = pd.read_sql_query(query, conn)
display(result)

,ward_name,total_calls
0,2V ZONE 2A,10434
1,2V ZONE 2B,8953
2,unknown,5786
3,1V ZONE 1A,4343
4,1V ZONE 1B,4119
5,GV ZONE KSW B,1463
6,GV ZONE KSW A,940
7,SERVICEFLATS,441
8,SERVICEFLATS 1V,14
9,SERVICEFLATS 2V,2
